In [ ]:
dbutils.widgets.text('Environment', 'dev', 'Set the current environment/catalog name')
dbutils.widgets.text('RunType', 'once', 'Set once to run as batch')
dbutils.widgets.text('ProcessingTime', '5 seconds', 'Set the microbatch interval')

In [ ]:
env = dbutils.widgets.get('Environment')
once = True if dbutils.widgets.get('RunType') == 'once' else False
processing_time = dbutils.widgets.get('ProcessingTime')
if once:
    print('Starting SBIT in batch mode')
else:
    print(f'Starting SBIT in streaming mode {processing_time} micro batch')

In [ ]:
spark.conf.set("spark.sql.shuffle.partitions", sc.defaultParallelism)
spark.conf.set("spark.databricks.delta.optimizeWrite.enabled", True)
spark.conf.set("spark.databricks.delta.autoCompact.enabled", True)
spark.conf.set("spark.sql.streaming.stateStore.providerClass", "com.databricks.sql.streaming.state.RocksDBStateStoreProvider")

In [ ]:
%run ./02-setup

In [ ]:
%run ./03-history-loader

In [ ]:
SH = SetupHelper(env)
HL = HistoryLoader(env)

In [ ]:
setup_required = spark.sql(f'show databases in {SH.catalog}').where(f'databaseName = "{SH.db_name}"').count() != 1
if setup_required:
    SH.setup()
    SH.validate()
    HL.load_history()
    HL.validate()
else:
    spark.sql(f'use {SH.catalog}.{SH.db_name}')

In [ ]:
%run ./04-bronze

In [ ]:
%run ./05-silver

In [ ]:
%run ./06-gold

In [ ]:
BZ = Bronze(env)
SL = Silver(env)
GL = Gold(env)

In [ ]:
BZ.consume(once, processing_time)

In [ ]:
SL.upsert(once, processing_time)

In [ ]:
GL.upsert(once, processing_time)